# Laboratorio 4: datos Sentinel-2 e índices espectrales

Este notebook conecta con la **Process API de Sentinel Hub**, consulta exclusivamente las fechas oficiales y genera para los lagos Atitlán y Amatitlán:

- estimación de clorofila-a asociada a cianobacteria (`cyanobacteria_chla`, µg/L);
- NDVI, usando B04 y B08;
- NDWI, usando B03 y B08;
- máscara de píxeles válidos de agua sin nube.

Los cálculos se realizan en Sentinel Hub y cada consulta descarga únicamente un GeoTIFF de cuatro bandas a 20 m, no una escena completa. El índice de cianobacteria reproduce el script oficial [Cyanobacteria Chlorophyll-a NDCI L1C](https://custom-scripts.sentinel-hub.com/sentinel-2/cyanobacteria_chla_ndci_l1c/), de Kravitz y Matthews (2020).

### Preparación

Se necesita una cuenta de Copernicus Data Space/Sentinel Hub y un cliente OAuth creado en el [Dashboard de Sentinel Hub](https://shapps.dataspace.copernicus.eu/dashboard/#/account/settings). El `client id` y el `client secret` se leen de variables de entorno o se solicitan de forma oculta; **no se escriben en el notebook ni en Git**.

In [1]:
%pip install -q sentinelhub rasterio geopandas shapely matplotlib pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datetime import date, timedelta
from getpass import getpass
from pathlib import Path
import json
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_bounds
from sentinelhub import (
    BBox, CRS, DataCollection, Geometry, MimeType, MosaickingOrder,
    SHConfig, SentinelHubRequest, SentinelHubSession, bbox_to_dimensions,
)

pd.set_option("display.max_rows", 30)

c:\Users\marti\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Conexión con la API de Sentinel-2

Se usa el endpoint de Sentinel Hub desplegado en Copernicus Data Space Ecosystem. La creación de una sesión valida las credenciales OAuth antes de solicitar imágenes.

In [6]:
config = SHConfig()
config.sh_client_id = os.getenv("SH_CLIENT_ID") or getpass("Sentinel Hub client id: ")
config.sh_client_secret = os.getenv("SH_CLIENT_SECRET") or getpass("Sentinel Hub client secret: ")
config.sh_base_url = "https://sh.dataspace.copernicus.eu"
config.sh_token_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

if not config.sh_client_id or not config.sh_client_secret:
    raise ValueError("Debe proporcionar SH_CLIENT_ID y SH_CLIENT_SECRET.")

session = SentinelHubSession(config=config)
token = session.token
print("Conexión OAuth establecida correctamente.")
print("El token vence en", token.get("expires_in", "?"), "segundos.")

InvalidClientError: (invalid_client) Invalid client or Invalid client credentials

## 2. Área, fechas y datos raster mínimos

Las coordenadas y las 11 fechas de cada lago son exactamente las proporcionadas. Se trabaja con Sentinel-2 **L1C**, producto para el cual fue calibrado el script oficial de cianobacteria. La resolución se fija en 20 m porque varias bandas necesarias para delimitar agua y detectar floración están disponibles originalmente a 20 m.

Si existen `data/atitlan.geojson` y `data/amatitlan.geojson`, se usan para recortar las solicitudes. De lo contrario se usan los rectángulos oficiales y el algoritmo enmascara tierra, nubes y píxeles sin datos.

In [ ]:
LAGOS = {
    "atitlan": {
        "bbox": [-91.326256, 14.5948, -91.07151, 14.750979],
        "geojson": Path("data/atitlan.geojson"),
        "fechas": [
            "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17",
            "2025-11-21", "2025-12-29", "2026-02-12", "2026-03-24",
            "2026-04-13", "2026-04-28", "2026-07-22",
        ],
    },
    "amatitlan": {
        "bbox": [-90.638065, 14.412347, -90.512924, 14.493799],
        "geojson": Path("data/amatitlan.geojson"),
        "fechas": [
            "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24",
            "2026-01-08", "2026-02-02", "2026-02-07", "2026-03-29",
            "2026-04-13", "2026-04-28", "2026-06-19",
        ],
    },
}

RESOLUCION_M = 20
DIRECTORIO_SALIDA = Path("data/processed")
DIRECTORIO_SALIDA.mkdir(parents=True, exist_ok=True)

registros = [
    {"lago": lago, "fecha": fecha}
    for lago, datos in LAGOS.items()
    for fecha in datos["fechas"]
]
fechas_df = pd.DataFrame(registros)
display(fechas_df.groupby("lago").agg(total=("fecha", "size"), inicio=("fecha", "min"), fin=("fecha", "max")))

In [ ]:
def cargar_area(nombre_lago):
    datos = LAGOS[nombre_lago]
    bbox = BBox(bbox=datos["bbox"], crs=CRS.WGS84)
    geometry = None

    if datos["geojson"].exists():
        gdf = gpd.read_file(datos["geojson"]).to_crs(4326)
        geom = gdf.geometry.union_all()
        geometry = Geometry(geom, crs=CRS.WGS84)
        bbox = geometry.bbox
        origen = str(datos["geojson"])
    else:
        origen = "coordenadas oficiales (bbox)"

    size = bbox_to_dimensions(bbox, resolution=RESOLUCION_M)
    return bbox, geometry, size, origen

for nombre in LAGOS:
    bbox, geometry, size, origen = cargar_area(nombre)
    print(f"{nombre.title():10s} | {size[0]} x {size[1]} píxeles | {origen}")

## 3. Cálculo en Sentinel Hub: cianobacteria, NDVI y NDWI

El Evalscript solicita solamente B02, B03, B04, B05, B08, B11 y B12, que permiten identificar agua y calcular NDCI/clorofila-a, NDVI y NDWI. `CLM` y `dataMask` se usan únicamente para control de calidad. El script oficial también colorea vegetación o material flotante mediante B07 y B8A; esas bandas no se descargan aquí porque no intervienen en el valor numérico de clorofila-a solicitado para el análisis posterior.

Fórmulas principales:

$$NDCI = \frac{B05-B04}{B05+B04}$$
$$Chl-a = 826.57NDCI^3 - 176.43NDCI^2 + 19NDCI + 4.071$$
$$NDVI = \frac{B08-B04}{B08+B04}, \qquad NDWI = \frac{B03-B08}{B03+B08}$$

Los valores fuera del agua, bajo nube o sin datos se escriben como `NaN`. La cuarta banda (`valid_mask`) vale 1 para observaciones válidas y 0 para las demás.

In [ ]:
EVALSCRIPT = r"""
//VERSION=3
function setup() {
  return {
    input: [{
      bands: ["B02", "B03", "B04", "B05", "B08", "B11", "B12", "CLM", "dataMask"],
      units: ["REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "REFLECTANCE", "DN", "DN"]
    }],
    output: { id: "indices", bands: 4, sampleType: "FLOAT32" }
  };
}

function ratio(a, b) {
  const denominator = a + b;
  return denominator === 0 ? NaN : (a - b) / denominator;
}

// Water Bodies Mapping, incluido en el CyanoLakes Custom Script oficial.
function isWater(s) {
  const ndvi = ratio(s.B08, s.B04);
  const mndwi = ratio(s.B03, s.B11);
  const ndwi = ratio(s.B03, s.B08);
  const ndwiLeaves = ratio(s.B08, s.B11);
  const aweish = s.B02 + 2.5 * s.B03 - 1.5 * (s.B08 + s.B11) - 0.25 * s.B12;
  const aweinsh = 4 * (s.B03 - s.B11) - (0.25 * s.B08 + 2.75 * s.B11);
  const dbsi = ratio(s.B11, s.B03) - ndvi;

  let water = mndwi > 0.42 || ndwi > 0.4 || aweinsh > 0.1879 ||
              aweish > 0.1112 || ndvi < -0.2 || ndwiLeaves > 1;
  if (water && (aweinsh <= -0.03 || dbsi > 0)) water = false;
  return water;
}

function evaluatePixel(s) {
  const valid = s.dataMask === 1 && s.CLM === 0 && isWater(s);
  if (!valid) return { indices: [NaN, NaN, NaN, 0] };

  const ndci = ratio(s.B05, s.B04);
  const chla = 826.57 * Math.pow(ndci, 3) - 176.43 * Math.pow(ndci, 2) + 19 * ndci + 4.071;
  const ndvi = ratio(s.B08, s.B04);
  const ndwi = ratio(s.B03, s.B08);
  return { indices: [chla, ndvi, ndwi, 1] };
}
"""

In [ ]:
NOMBRES_BANDAS = ("cyanobacteria_chla", "ndvi", "ndwi", "valid_mask")
S2L1C_CDSE = DataCollection.SENTINEL2_L1C.define_from(
    "s2l1c_cdse", service_url=config.sh_base_url
)

def intervalo_de_un_dia(fecha_iso):
    inicio = date.fromisoformat(fecha_iso)
    fin = inicio + timedelta(days=1)
    return inicio.isoformat(), fin.isoformat()

def descargar_indices(nombre_lago, fecha_iso, sobrescribir=False):
    salida = DIRECTORIO_SALIDA / nombre_lago / f"{nombre_lago}_{fecha_iso}_indices.tif"
    salida.parent.mkdir(parents=True, exist_ok=True)
    if salida.exists() and not sobrescribir:
        return salida, "existente"

    bbox, geometry, size, _ = cargar_area(nombre_lago)
    area_kwargs = {"geometry": geometry} if geometry is not None else {"bbox": bbox}
    request = SentinelHubRequest(
        evalscript=EVALSCRIPT,
        input_data=[SentinelHubRequest.input_data(
            data_collection=S2L1C_CDSE,
            time_interval=intervalo_de_un_dia(fecha_iso),
            mosaicking_order=MosaickingOrder.LEAST_CC,
        )],
        responses=[SentinelHubRequest.output_response("indices", MimeType.TIFF)],
        size=size,
        config=config,
        **area_kwargs,
    )
    imagen = request.get_data()[0]
    if imagen.ndim != 3 or imagen.shape[-1] != 4:
        raise RuntimeError(f"Respuesta inesperada para {nombre_lago} {fecha_iso}: {imagen.shape}")

    transform = from_bounds(*bbox, width=imagen.shape[1], height=imagen.shape[0])
    with rasterio.open(
        salida, "w", driver="GTiff", height=imagen.shape[0], width=imagen.shape[1],
        count=4, dtype="float32", crs="EPSG:4326", transform=transform,
        compress="deflate", predictor=3, tiled=True, nodata=np.nan,
    ) as dst:
        for banda, nombre in enumerate(NOMBRES_BANDAS, start=1):
            dst.write(imagen[:, :, banda - 1].astype("float32"), banda)
            dst.set_band_description(banda, nombre)
        dst.update_tags(lago=nombre_lago, fecha=fecha_iso, fuente="Sentinel-2 L1C / Sentinel Hub Process API")

    return salida, "descargado"

### Descarga de las 22 imágenes seleccionadas

La celda siguiente puede reanudarse: omite cualquier archivo ya creado. Para una prueba rápida, cambie `MODO_PRUEBA` a `True`; esto procesa solamente la primera fecha de cada lago.

In [ ]:
MODO_PRUEBA = False
resultados = []

for nombre_lago, datos in LAGOS.items():
    fechas = datos["fechas"][:1] if MODO_PRUEBA else datos["fechas"]
    for fecha_iso in fechas:
        try:
            archivo, estado = descargar_indices(nombre_lago, fecha_iso)
            resultados.append({"lago": nombre_lago, "fecha": fecha_iso, "estado": estado, "archivo": str(archivo)})
            print(f"[OK] {nombre_lago:10s} {fecha_iso}: {estado}")
        except Exception as exc:
            resultados.append({"lago": nombre_lago, "fecha": fecha_iso, "estado": "error", "archivo": str(exc)})
            print(f"[ERROR] {nombre_lago:10s} {fecha_iso}: {exc}")

resultados_df = pd.DataFrame(resultados)
display(resultados_df)

### Control de calidad y visualización

Se verifica que cada archivo tenga las cuatro bandas esperadas y se reporta el porcentaje del rectángulo solicitado clasificado como agua válida y sin nube. Esta métrica no equivale a la cobertura oficial de la escena porque también excluye la tierra incluida en el rectángulo. Los mapas permiten confirmar visualmente que los tres productos fueron generados para cada imagen seleccionada.

In [ ]:
def inspeccionar_archivo(ruta):
    with rasterio.open(ruta) as src:
        mascara = src.read(4) == 1
        return {
            "archivo": str(ruta),
            "ancho": src.width,
            "alto": src.height,
            "bandas": src.descriptions,
            "pixeles_validos": int(mascara.sum()),
            "cobertura_valida_bbox_pct": round(100 * mascara.mean(), 2),
        }

archivos = sorted(DIRECTORIO_SALIDA.glob("*/*_indices.tif"))
control_df = pd.DataFrame(inspeccionar_archivo(ruta) for ruta in archivos)
display(control_df)

In [ ]:
def visualizar_indices(ruta):
    with rasterio.open(ruta) as src:
        chla, ndvi, ndwi, mascara = src.read()
        titulo = f"{src.tags().get('lago', '')} | {src.tags().get('fecha', '')}"

    validos = mascara == 1
    capas = [
        (np.where(validos, chla, np.nan), "Cianobacteria (Chl-a, µg/L)", "turbo", None, None),
        (np.where(validos, ndvi, np.nan), "NDVI", "RdYlGn", -1, 1),
        (np.where(validos, ndwi, np.nan), "NDWI", "Blues", -1, 1),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    for ax, (capa, nombre, cmap, vmin, vmax) in zip(axes, capas):
        imagen = ax.imshow(capa, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(nombre)
        ax.axis("off")
        fig.colorbar(imagen, ax=ax, shrink=0.75)
    fig.suptitle(titulo, fontsize=14)
    plt.show()

for ruta in archivos:
    visualizar_indices(ruta)

## Resultado de los incisos 1–3

1. La sesión OAuth demuestra la conexión funcional con la API de Sentinel-2.
2. Cada archivo de `data/processed/<lago>/` corresponde a una fecha oficial y contiene únicamente los índices requeridos y su máscara, calculados a 20 m.
3. Los mapas muestran el producto CyanoLakes de clorofila-a/cianobacteria, NDVI y NDWI. Estos GeoTIFF quedan listos para los análisis temporal, espacial y de correlación de los incisos posteriores.

**Limitación:** el producto estima clorofila-a a partir de reflectancia y funciona como indicador remoto de posibles floraciones; no confirma por sí solo especie, toxicidad ni concentración medida en campo. Nubes, sombras, sedimentos y vegetación flotante pueden afectar la estimación.